In [10]:
import numpy as np
import pandas as pd
import plotly.express as px

from acdesign.airfoils.polar import UIUCPolar
from acdesign.atmosphere import Atmosphere
from acdesign.performance.aero import AircraftAero, FuseAero, WingAero
from acdesign.solar_wing import SolarWing

sg6041 = UIUCPolar.local("SG6041")
e472 = UIUCPolar.local("E472")

atm = Atmosphere.alt(0)
airspeed = np.linspace(7, 25, 42)
mass = 4.5
solarwing = SolarWing.straight_to_elliptical(19, 18, 2, sg6041)
wing = solarwing.wing
fus_length = 1.5

aircraft = AircraftAero(
    wing,
    WingAero(wing.b * 0.2, wing.S * 0.2, [e472], [0, 1]),
    WingAero(wing.b * 0.1, wing.S * 0.1, [e472], [0, 1]),
    FuseAero(fus_length, 0.15),
    0.02,
    fus_length * 0.75,
)

aircraft.trim(
    atm,
    airspeed,
    mass * 9.81,
)


TypeError: unsupported operand type(s) for &: 'tuple' and 'int'

In [2]:
lift = np.full_like(airspeed, mass * 9.81)

cls = wing.aero.get_cl(atm, airspeed, lift)

loads, sloads = wing.wing.run_avl(
    cls, ylocs=np.linspace(0, 1, 50), sections=[wing.aero.polars[0]] * 50
)


wing_results = wing.aero(atm, airspeed, lift, sloads[-1], n=50, mode="oto")

Cd = loads.CDind + wing_results.Cd0


In [3]:
from acdesign.performance.aero import FuseAero

pod = FuseAero(0.8, 0.15)
pylons = FuseAero(1.5, 0.07)


def fus_drag(component: FuseAero):
    res = component(atm, airspeed, loads.Alpha)
    Cd = res.Cd0 * res.S / wing.wing.S
    return Cd


Cd_fus = fus_drag(pod) + fus_drag(pylons) * 2


In [4]:
wing_results = wing_results.assign(Cd_fus=Cd_fus, Cd_ind=loads.CDind)
wing_results.to_csv("examples/solar_plane/wing_results.csv")


In [ ]:
from json import dumps

# px.line(loads,x="Alpha", y="CLff")
_lmax = loads.loc[loads.Cl == loads.Cl.max()]
_lmin = loads.loc[loads.Cl == loads.Cl.min()]

_clmax = _lmax.Cl.values[0]
_clmin = _lmin.Cl.values[0]
_amax = _lmax.Alpha.values[0]
_amin = _lmin.Alpha.values[0]

dclda = (_clmax - _clmin) / (_amax - _amin)
a0 = _amax - _clmax / dclda

dumps(dict(dclda=dclda, a0=a0))

'{"dclda": 0.09660905893236313, "a0": -2.621460989583335}'

In [ ]:
_arsp = np.linspace(10, 24, 8)
_thrust = np.linspace(5, 40, 100)

_ta = np.meshgrid(_arsp, _thrust)
_arsp, _thrust = _ta[0].reshape(-1), _ta[1].reshape(-1)

_power = wing.propulsion(atm.alt(0), _arsp, _thrust)

df = pd.DataFrame({"airspeed": _arsp, "thrust": _thrust, "power": _power})
df.to_csv("examples/solar_plane/propulsion_prediction.csv", index=False)
df

,airspeed,thrust,power
0,10.0,5.0,153.846154
1,12.0,5.0,184.615385
2,14.0,5.0,215.384615
3,16.0,5.0,246.153846
4,18.0,5.0,276.923077
...,...,...,...
795,16.0,40.0,1969.230769
796,18.0,40.0,2215.384615
797,20.0,40.0,2461.538462
798,22.0,40.0,2707.692308
